In [1]:
import time
from pyspark.sql import SparkSession
import mysql.connector
from pyspark.sql.functions import col

MYSQL BENCHMARK


In [8]:
MYSQL_CONFIG = {
    "host": "localhost",
    "port": "3306",
    "user": "root",
    "password": "root",
    "database": "tpch",
}
conn = mysql.connector.connect(**MYSQL_CONFIG)
cursor = conn.cursor()  

(cursor.execute
 ("SELECT VERSION()"))
print("Connected to MySQL:", cursor.fetchone()[0])


Connected to MySQL: 8.0.44


In [3]:

def execute_query_with_timing(query, cursor):
    start_time = time.time()
    cursor.execute(query)
    results = cursor.fetchall()
    end_time = time.time()
    
    execution_time = end_time - start_time
    
    return execution_time, results


In [ ]:
mysql_query = """
SELECT
    n_name,
    s_name,
    SUM(l_quantity) AS sum_qty,
    SUM(l_extendedprice) AS sum_base_price,
    SUM(l_extendedprice * (1 - l_discount)) AS sum_disc_price,
    SUM(l_extendedprice * (1 - l_discount) * (1 + l_tax)) AS sum_charge,
    AVG(l_quantity) AS avg_qty,
    AVG(l_extendedprice) AS avg_price,
    AVG(l_discount) AS avg_disc,
    COUNT(*) AS count_order
FROM
    Lineitem,
    Orders,
    Customer,
    Nation,     
    Supplier
WHERE
    l_shipdate <= DATE '1998-12-01' - INTERVAL 90 DAY
    AND l_orderkey = o_orderkey
    AND o_custkey = c_custkey
    AND c_nationkey = n_nationkey
    AND l_suppkey = s_suppkey
GROUP BY
    n_name,
    s_name;"""




In [10]:
execution_times = []
for i in range(8):
    exec_time, results = execute_query_with_timing(mysql_query, cursor)
    execution_times.append(exec_time)
    print(f"Execution {i+1}: {exec_time:.4f} seconds")

print("Average execution time:", sum(execution_times) / len(execution_times), "seconds")


Execution 1: 83.4452 seconds
Execution 2: 82.3609 seconds
Execution 3: 86.4826 seconds
Execution 4: 84.9024 seconds
Execution 5: 85.2007 seconds
Execution 6: 82.1944 seconds
Execution 7: 81.5020 seconds
Execution 8: 83.0566 seconds
Average execution time: 83.64308789372444 seconds


In [ ]:

cursor.close()
conn.close()

Spark benchmark


In [2]:
spark = SparkSession.builder \
    .appName("TPCH") \
    .config("spark.executor.memory", "8g") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/09 07:48:57 WARN Utils: Your hostname, oden-Valhalla, resolves to a loopback address: 127.0.1.1; using 10.92.66.73 instead (on interface wlp3s0)
26/03/09 07:48:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 07:48:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


read dim customer table


In [3]:
dim_customer_df = spark.read.parquet("../spark/data/dim_customer")


print(f"Total records: {dim_customer_df.count()}")

print("\nSchema:")
dim_customer_df.printSchema()

# Show first 20 rows
print("\nFirst 20 rows:")
dim_customer_df.show(20, truncate=False)

Total records: 2550000

Schema:
root
 |-- cust_key: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- acct_bal: string (nullable = true)
 |-- mkt_segment: string (nullable = true)
 |-- nation_name: string (nullable = true)
 |-- region_name: string (nullable = true)


First 20 rows:
+--------+------------------+--------------------------------------+---------------+--------+-----------+--------------+-----------+
|cust_key|customer_name     |address                               |phone          |acct_bal|mkt_segment|nation_name   |region_name|
+--------+------------------+--------------------------------------+---------------+--------+-----------+--------------+-----------+
|1       |Customer#000000001|IVhzIApeRb ot,c,E                     |25-989-741-2988|711.56  |BUILDING   |MOROCCO       |AFRICA     |
|2       |Customer#000000002|XSTf4,NCwDVaWNe6tEgvwfmRchLXak        |23-768-687-3665

read dim supplier table

In [4]:
dim_supplier_df = spark.read.parquet("../spark/data/dim_supplier")

print(f"Total records: {dim_supplier_df.count()}")

print("\nSchema:")
dim_supplier_df.printSchema()

print("\nFirst 20 rows:")
dim_supplier_df.show(20, truncate=False)

Total records: 30000

Schema:
root
 |-- supp_key: integer (nullable = true)
 |-- supplier_name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- acct_bal: string (nullable = true)
 |-- nation_name: string (nullable = true)
 |-- region_name: string (nullable = true)


First 20 rows:
+--------+------------------+------------------------------------+---------------+--------+--------------+-----------+
|supp_key|supplier_name     |address                             |phone          |acct_bal|nation_name   |region_name|
+--------+------------------+------------------------------------+---------------+--------+--------------+-----------+
|1       |Supplier#000000001| N kD4on9OM Ipw3,gf0JBoQDd7tgrzrddZ |27-918-335-1736|5755.94 |PERU          |AMERICA    |
|2       |Supplier#000000002|89eJ5ksX3ImxJQBvxObC,               |15-679-861-2259|4032.68 |ETHIOPIA      |AFRICA     |
|3       |Supplier#000000003|q1,G3Pj6OjIuUYfUoH18BFTKP5aU9bEV3   

read fact sales table

In [5]:
fact_sales_df = spark.read.parquet("../spark/data/fact_sales")
fact_sales_df = fact_sales_df.select(
    col("cust_key"),
    col("supp_key"),
    col("quantity").cast("double").alias("quantity"),
    col("extended_price").cast("double").alias("extended_price"),
    col("discount").cast("double").alias("discount"),
    col("tax").cast("double").alias("tax"),
    col("ship_date")
)
print(f"Total records: {fact_sales_df.count()}")

print("\nSchema:")
fact_sales_df.printSchema()

print("\nFirst 20 rows:")
fact_sales_df.show(20, truncate=False)

Total records: 24004860

Schema:
root
 |-- cust_key: integer (nullable = true)
 |-- supp_key: integer (nullable = true)
 |-- quantity: double (nullable = true)
 |-- extended_price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- tax: double (nullable = true)
 |-- ship_date: string (nullable = true)


First 20 rows:
+--------+--------+--------+--------------+--------+----+------------+
|cust_key|supp_key|quantity|extended_price|discount|tax |ship_date   |
+--------+--------+--------+--------------+--------+----+------------+
|1       |5824    |40.0    |51771.6       |0.05    |0.05|709344000000|
|1       |487     |13.0    |25011.61      |0.1     |0.02|704764800000|
|1       |7128    |33.0    |53513.13      |0.0     |0.01|859680000000|
|1       |8384    |36.0    |48649.68      |0.04    |0.05|720316800000|
|1       |3180    |9.0     |14891.76      |0.05    |0.03|720144000000|
|1       |2273    |32.0    |54680.32      |0.07    |0.01|722995200000|
|1       |9400    |6.0

preparing tables and  SQL query

In [6]:
dim_customer_df.createOrReplaceTempView("dim_customer")
dim_supplier_df.createOrReplaceTempView("dim_supplier")
fact_sales_df.createOrReplaceTempView("fact_sales")



In [7]:
query = """
SELECT
    c.nation_name,
    s.supplier_name,
    SUM(f.quantity) AS sum_qty,
    SUM(f.extended_price) AS sum_base_price,
    SUM(f.extended_price * (1 - f.discount)) AS sum_disc_price,
    SUM(f.extended_price * (1 - f.discount) * (1 + f.tax)) AS sum_charge,
    AVG(f.quantity) AS avg_qty,
    AVG(f.extended_price) AS avg_price,
    AVG(f.discount) AS avg_disc,
    COUNT(*) AS count_order
FROM fact_sales f
JOIN dim_customer c ON f.cust_key = c.cust_key
JOIN dim_supplier s ON f.supp_key = s.supp_key
WHERE f.ship_date <= 906307200000
GROUP BY c.nation_name, s.supplier_name
"""

Executing the query and showing results

In [8]:
import time

spark_execution_time = []
num_iterations = 8



print(f"Executing query {num_iterations} measured times...\n")

for i in range(num_iterations):
    print(f"Iteration {i + 1}/{num_iterations}:")
    start_time = time.perf_counter()

    result_count = spark.sql(query)
    result_count.show(100) 
    end_time = time.perf_counter()
    execution_time = end_time - start_time
    spark_execution_time.append(execution_time)

    print(f"  Result rows: {result_count}")
    print(f"  Execution time: {execution_time:.4f} seconds\n")

average_time = sum(spark_execution_time) / len(spark_execution_time)
min_time = min(spark_execution_time)
max_time = max(spark_execution_time)

print(f"\nAverage execution time: {average_time:.4f} s")
print(f"Min execution time: {min_time:.4f} s")
print(f"Max execution time: {max_time:.4f} s")




Executing query 8 measured times...

Iteration 1/8:


+--------------+------------------+--------+--------------------+--------------------+--------------------+------------------+------------------+--------------------+-----------+
|   nation_name|     supplier_name| sum_qty|      sum_base_price|      sum_disc_price|          sum_charge|           avg_qty|         avg_price|            avg_disc|count_order|
+--------------+------------------+--------+--------------------+--------------------+--------------------+------------------+------------------+--------------------+-----------+
|UNITED KINGDOM|Supplier#000001057|124236.0|1.6169543675999957E8|1.5344633188559997E8|1.5952563585052752E8|            25.375| 33026.02874999991|0.052083333333333405|       4896|
|       MOROCCO|Supplier#000007393| 61812.0|1.0611911903999983E8|1.0041932843999979E8|1.0404916416837634E8|           18.9375|32511.984999999946| 0.04875000000000013|       3264|
|       MOROCCO|Supplier#000007299|185436.0| 2.959458722399998E8|2.8042318659479994E8| 2.928877070325247E

+--------------+------------------+--------+--------------------+--------------------+--------------------+------------------+------------------+--------------------+-----------+
|   nation_name|     supplier_name| sum_qty|      sum_base_price|      sum_disc_price|          sum_charge|           avg_qty|         avg_price|            avg_disc|count_order|
+--------------+------------------+--------+--------------------+--------------------+--------------------+------------------+------------------+--------------------+-----------+
|UNITED KINGDOM|Supplier#000001057|124236.0|1.6169543675999957E8|1.5344633188559997E8|1.5952563585052752E8|            25.375| 33026.02874999991|0.052083333333333405|       4896|
|       MOROCCO|Supplier#000007393| 61812.0|1.0611911903999983E8|1.0041932843999979E8|1.0404916416837634E8|           18.9375|32511.984999999946| 0.04875000000000013|       3264|
|       MOROCCO|Supplier#000007299|185436.0| 2.959458722399998E8|2.8042318659479994E8| 2.928877070325247E

+--------------+------------------+--------+--------------------+--------------------+--------------------+------------------+------------------+--------------------+-----------+
|   nation_name|     supplier_name| sum_qty|      sum_base_price|      sum_disc_price|          sum_charge|           avg_qty|         avg_price|            avg_disc|count_order|
+--------------+------------------+--------+--------------------+--------------------+--------------------+------------------+------------------+--------------------+-----------+
|UNITED KINGDOM|Supplier#000001057|124236.0|1.6169543675999957E8|1.5344633188559997E8|1.5952563585052752E8|            25.375| 33026.02874999991|0.052083333333333405|       4896|
|       MOROCCO|Supplier#000007393| 61812.0|1.0611911903999983E8|1.0041932843999979E8|1.0404916416837634E8|           18.9375|32511.984999999946| 0.04875000000000013|       3264|
|       MOROCCO|Supplier#000007299|185436.0| 2.959458722399998E8|2.8042318659479994E8| 2.928877070325247E

+--------------+------------------+--------+--------------------+--------------------+--------------------+------------------+------------------+--------------------+-----------+
|   nation_name|     supplier_name| sum_qty|      sum_base_price|      sum_disc_price|          sum_charge|           avg_qty|         avg_price|            avg_disc|count_order|
+--------------+------------------+--------+--------------------+--------------------+--------------------+------------------+------------------+--------------------+-----------+
|UNITED KINGDOM|Supplier#000001057|124236.0|1.6169543675999957E8|1.5344633188559997E8|1.5952563585052752E8|            25.375| 33026.02874999991|0.052083333333333405|       4896|
|       MOROCCO|Supplier#000007393| 61812.0|1.0611911903999983E8|1.0041932843999979E8|1.0404916416837634E8|           18.9375|32511.984999999946| 0.04875000000000013|       3264|
|       MOROCCO|Supplier#000007299|185436.0| 2.959458722399998E8|2.8042318659479994E8| 2.928877070325247E

+--------------+------------------+--------+--------------------+--------------------+--------------------+------------------+------------------+--------------------+-----------+
|   nation_name|     supplier_name| sum_qty|      sum_base_price|      sum_disc_price|          sum_charge|           avg_qty|         avg_price|            avg_disc|count_order|
+--------------+------------------+--------+--------------------+--------------------+--------------------+------------------+------------------+--------------------+-----------+
|UNITED KINGDOM|Supplier#000001057|124236.0|1.6169543675999957E8|1.5344633188559997E8|1.5952563585052752E8|            25.375| 33026.02874999991|0.052083333333333405|       4896|
|       MOROCCO|Supplier#000007393| 61812.0|1.0611911903999983E8|1.0041932843999979E8|1.0404916416837634E8|           18.9375|32511.984999999946| 0.04875000000000013|       3264|
|       MOROCCO|Supplier#000007299|185436.0| 2.959458722399998E8|2.8042318659479994E8| 2.928877070325247E

+--------------+------------------+--------+--------------------+--------------------+--------------------+------------------+------------------+--------------------+-----------+
|   nation_name|     supplier_name| sum_qty|      sum_base_price|      sum_disc_price|          sum_charge|           avg_qty|         avg_price|            avg_disc|count_order|
+--------------+------------------+--------+--------------------+--------------------+--------------------+------------------+------------------+--------------------+-----------+
|UNITED KINGDOM|Supplier#000001057|124236.0|1.6169543675999957E8|1.5344633188559997E8|1.5952563585052752E8|            25.375| 33026.02874999991|0.052083333333333405|       4896|
|       MOROCCO|Supplier#000007393| 61812.0|1.0611911903999983E8|1.0041932843999979E8|1.0404916416837634E8|           18.9375|32511.984999999946| 0.04875000000000013|       3264|
|       MOROCCO|Supplier#000007299|185436.0| 2.959458722399998E8|2.8042318659479994E8| 2.928877070325247E

+--------------+------------------+--------+--------------------+--------------------+--------------------+------------------+------------------+--------------------+-----------+
|   nation_name|     supplier_name| sum_qty|      sum_base_price|      sum_disc_price|          sum_charge|           avg_qty|         avg_price|            avg_disc|count_order|
+--------------+------------------+--------+--------------------+--------------------+--------------------+------------------+------------------+--------------------+-----------+
|UNITED KINGDOM|Supplier#000001057|124236.0|1.6169543675999957E8|1.5344633188559997E8|1.5952563585052752E8|            25.375| 33026.02874999991|0.052083333333333405|       4896|
|       MOROCCO|Supplier#000007393| 61812.0|1.0611911903999983E8|1.0041932843999979E8|1.0404916416837634E8|           18.9375|32511.984999999946| 0.04875000000000013|       3264|
|       MOROCCO|Supplier#000007299|185436.0| 2.959458722399998E8|2.8042318659479994E8| 2.928877070325247E

+--------------+------------------+--------+--------------------+--------------------+--------------------+------------------+------------------+--------------------+-----------+
|   nation_name|     supplier_name| sum_qty|      sum_base_price|      sum_disc_price|          sum_charge|           avg_qty|         avg_price|            avg_disc|count_order|
+--------------+------------------+--------+--------------------+--------------------+--------------------+------------------+------------------+--------------------+-----------+
|UNITED KINGDOM|Supplier#000001057|124236.0|1.6169543675999957E8|1.5344633188559997E8|1.5952563585052752E8|            25.375| 33026.02874999991|0.052083333333333405|       4896|
|       MOROCCO|Supplier#000007393| 61812.0|1.0611911903999983E8|1.0041932843999979E8|1.0404916416837634E8|           18.9375|32511.984999999946| 0.04875000000000013|       3264|
|       MOROCCO|Supplier#000007299|185436.0| 2.959458722399998E8|2.8042318659479994E8| 2.928877070325247E